In [1]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', None)

Load merged data

In [3]:
df = pd.read_csv('merged_streaming_data.csv')
df.head()

,id_student,code_module,code_presentation,date,forumng,homepage,oucontent,subpage,url,resource,glossary,dataplus,oucollaborate,quiz,ouelluminate,sharedsubpage,questionnaire,page,externalquiz,ouwiki,dualpane,repeatactivity,folder,htmlactivity,id_assessment,is_banked,score,assessment_type,due_date,weight,module_presentation_length,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration
0,6516,AAA,2014J,-23.0,0,3,23,2,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
1,6516,AAA,2014J,-22.0,33,13,34,0,0,2,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
2,6516,AAA,2014J,-20.0,13,12,8,1,0,7,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
3,6516,AAA,2014J,-17.0,0,2,0,3,2,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN
4,6516,AAA,2014J,-12.0,1,1,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN


In [4]:
# Create a "week" column relative to each code_presentation's starting date. 
# Dates 0.0 - 6.0 are week 1, 7.0 - 13.0 are week 2, etc. 
# The weeks *before* date 0.0 are assigned to negative values.

df['week'] = df.groupby('code_presentation')['date'].transform(
    lambda x: ((x // 7) + 1).where(x >= 0, x // 7)
)

In [5]:
df.columns

Index(['id_student', 'code_module', 'code_presentation', 'date', 'forumng',
       'homepage', 'oucontent', 'subpage', 'url', 'resource', 'glossary',
       'dataplus', 'oucollaborate', 'quiz', 'ouelluminate', 'sharedsubpage',
       'questionnaire', 'page', 'externalquiz', 'ouwiki', 'dualpane',
       'repeatactivity', 'folder', 'htmlactivity', 'id_assessment',
       'is_banked', 'score', 'assessment_type', 'due_date', 'weight',
       'module_presentation_length', 'gender', 'region', 'highest_education',
       'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits',
       'disability', 'final_result', 'date_registration',
       'date_unregistration', 'week'],
      dtype='object')

In [6]:
# Number of id_students only in one module-presentation, or more
df[['id_student', 'code_module', 'code_presentation']].drop_duplicates() \
  .groupby('id_student').size().value_counts().sort_index()

1    23088
2     2856
3      143
4       11
5        1
Name: count, dtype: int64

In [7]:
course_student_vle_cols = df.columns[:24].tolist() + df.columns[30:].tolist()
course_student_assessment_cols = df.columns[:4].tolist() + df.columns[24:].tolist()

vle_columns = ['quiz', 'questionnaire', 'externalquiz', 'oucontent', 'page', 'resource', 'url', 'homepage', 
               'glossary', 'subpage', 'folder', 'forumng', 'oucollaborate', 'ouelluminate', 'ouwiki', 'sharedsubpage', 
               'dataplus', 'repeatactivity', 'dualpane', 'htmlactivity'
]

In [8]:
# Separate dataframe into vle and assessment relevant dataframes (with student and course info in both)

# Keep columns relevant to assessments and drop rows with no assessment 
df_assessment = df[course_student_assessment_cols].dropna(subset=["id_assessment"])

# Keep columns relevant to vle interactions and drop duplicate rows that arose if students submitted multiple assessments on a given date
df_vle = df[course_student_vle_cols].drop_duplicates()
# Keep rows where at least one VLE column is nonzero and not NaN
df_vle = df_vle[df_vle[vle_columns].fillna(0).sum(axis=1) > 0] # takes care of instances where there is NaN in a column (I noticed it in quiz) and everything else is 0

# there should be no 'date' duplicated for each (id_student,code_module,code_presentation) combination
# check: 
has_duplicates = df_vle.duplicated(
    subset=["id_student", "code_module", "code_presentation", "date"]
).any()
print(has_duplicates) # should be False

False


Keep only first assessment metrics 

Determine the first assessment for each module-presentation and only keep that assessment for each student

In [9]:
# Use OULAD assessments.csv to easily identify first assessment for each module, presentation combination
assessments = pd.read_csv('data/OULAD/assessments.csv')

# Rename 'date' column in assessments to 'due_date'
assessments = assessments.rename(columns={'date': 'due_date'})
assessments.columns

Index(['code_module', 'code_presentation', 'id_assessment', 'assessment_type',
       'due_date', 'weight'],
      dtype='object')

assessments.csv<br>
This file contains information about assessments in module-presentations. Usually, every presentation has a number of assessments followed by the final exam. CSV contains columns:<br>
- *code_module* - identification code of the module, to which the assessment belongs.<br>
- *code_presentation* - identification code of the presentation, to which the assessment belongs.<br>
- *id_assessment* - identification number of the assessment.<br>
- *assessment_type* - type of assessment. Three types of assessments exist: Tutor Marked Assessment (TMA), Computer Marked Assessment (CMA) and Final Exam (Exam).<br>
- *date* - information about the final submission date of the assessment calculated as the number of days since the start of the module-presentation. The starting date of the presentation has number 0 (zero).<br>
- *weight* - weight of the assessment in %. Typically, Exams are treated separately and have the weight 100%; the sum of all other assessments is 100%.<br>

If the information about the final exam date is missing, it is at the end of the last presentation week.<br>
<br>

In [10]:
# Determine what is the first assessment per module, presentation
first_assessment = (
    assessments
    .sort_values('due_date', ascending=True)  # sort so earliest comes first
    .groupby(['code_module', 'code_presentation'], as_index=False)
    .first()  # keep the first row per group (i.e., earliest due_date)
)
first_assessment

,code_module,code_presentation,id_assessment,assessment_type,due_date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2014J,1758,TMA,19.0,10.0
2,BBB,2013B,14984,TMA,19.0,5.0
3,BBB,2013J,14996,TMA,19.0,5.0
4,BBB,2014B,15008,TMA,12.0,5.0
5,BBB,2014J,15020,TMA,19.0,0.0
6,CCC,2014B,24286,CMA,18.0,2.0
7,CCC,2014J,24295,CMA,18.0,2.0
8,DDD,2013B,25341,CMA,23.0,2.0
9,DDD,2013J,25348,TMA,25.0,10.0


In [11]:
# Determine which modules where all presentations have a first assessment in the first 3 weeks (<21 days)
idx = (
    first_assessment
    .groupby('code_module')['due_date']
    .max()                    # find the max due_date per module
    .loc[lambda x: x < 21]    # keep only modules whose max due_date < 21
    .index
)
modules_to_keep = first_assessment[first_assessment['code_module'].isin(idx)]
modules_to_keep

,code_module,code_presentation,id_assessment,assessment_type,due_date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2014J,1758,TMA,19.0,10.0
2,BBB,2013B,14984,TMA,19.0,5.0
3,BBB,2013J,14996,TMA,19.0,5.0
4,BBB,2014B,15008,TMA,12.0,5.0
5,BBB,2014J,15020,TMA,19.0,0.0
6,CCC,2014B,24286,CMA,18.0,2.0
7,CCC,2014J,24295,CMA,18.0,2.0


In [12]:
# Only keep first assessment rows for each student
df_first_assessment = df_assessment.merge(
    first_assessment[['code_module', 'code_presentation', 'id_assessment']],
    on=['code_module', 'code_presentation', 'id_assessment'],
    how='inner'
)
df_first_assessment = df_first_assessment.rename(columns={'due_date': 'first_due_date'}) # rename column
df_first_assessment.head()

,id_student,code_module,code_presentation,date,id_assessment,is_banked,score,assessment_type,first_due_date,weight,module_presentation_length,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,week
0,6516,AAA,2014J,17.0,1758.0,0.0,60.0,TMA,19.0,10.0,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,3.0
1,8462,DDD,2013J,29.0,25348.0,0.0,93.0,TMA,25.0,10.0,261,M,London Region,HE Qualification,30-40%,55<=,0,90,N,Withdrawn,-137.0,119.0,5.0
2,8462,DDD,2014J,-1.0,25362.0,1.0,93.0,TMA,20.0,5.0,262,M,London Region,HE Qualification,30-40%,55<=,1,60,N,Withdrawn,-38.0,18.0,-1.0
3,11391,AAA,2013J,18.0,1752.0,0.0,78.0,TMA,19.0,10.0,268,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN,3.0
4,23629,BBB,2013B,9.0,14984.0,0.0,67.0,TMA,19.0,5.0,240,F,East Anglian Region,Lower Than A Level,20-30%,0-35,2,60,N,Fail,-47.0,NaN,2.0


In [13]:
# Perform some checks on unique combinations of id_student, code_module, code_presentation 

# Count unique student-module-presentation combinations
print(df[['id_student', 'code_module', 'code_presentation']].drop_duplicates().shape[0], "unique combinations in df (includes both VLE interactions and assessments)")
print(df_assessment[['id_student', 'code_module', 'code_presentation']].drop_duplicates().shape[0], "unique combinations in df_assessment (includes all assessments)")
print(df_first_assessment[['id_student', 'code_module', 'code_presentation']].drop_duplicates().shape[0], "unique combinations in df_first_assessment (only first assessments)")
print(df_vle[['id_student', 'code_module', 'code_presentation']].drop_duplicates().shape[0], "unique combinations in df_vle (includes VLE interactions)")

# Now compute the differences
num_no_assessment = (
    df_vle[['id_student', 'code_module', 'code_presentation']].drop_duplicates()
    .merge(df_assessment[['id_student', 'code_module', 'code_presentation']].drop_duplicates(),
           on=['id_student', 'code_module', 'code_presentation'], how='left', indicator=True)
    .query('_merge == "left_only"')
    .shape[0]
)

num_no_first_assessment = (
    df_vle[['id_student', 'code_module', 'code_presentation']].drop_duplicates()
    .merge(df_first_assessment[['id_student', 'code_module', 'code_presentation']].drop_duplicates(),
           on=['id_student', 'code_module', 'code_presentation'], how='left', indicator=True)
    .query('_merge == "left_only"')
    .shape[0]
)

print(num_no_assessment, "student–module–presentation combinations did not turn in any assessment.")
print(num_no_first_assessment, "student–module–presentation combinations did not turn in the first assessment.")

29278 unique combinations in df (includes both VLE interactions and assessments)
25843 unique combinations in df_assessment (includes all assessments)
25385 unique combinations in df_first_assessment (only first assessments)
29228 unique combinations in df_vle (includes VLE interactions)
3435 student–module–presentation combinations did not turn in any assessment.
3890 student–module–presentation combinations did not turn in the first assessment.


In [14]:
# Add students who did not turn in a first assessment back into df_first_assessment and following assessment-related columns will be NaN:
# - date	
# - is_banked	
# - score	
# - week

# Step 1: Start with all unique students from df_vle (keep all student info columns)
students_all = df_vle[[
    'id_student', 'code_module', 'code_presentation',
    'gender', 'region', 'highest_education', 'imd_band', 'age_band',
    'num_of_prev_attempts', 'studied_credits', 'disability',
    'final_result', 'date_registration', 'date_unregistration',
    'module_presentation_length'
]].drop_duplicates()

# Step 2: Merge with df_first_assessment (left join keeps all VLE students)
df_first_assessment_full = students_all.merge(
    df_first_assessment,
    on=['id_student', 'code_module', 'code_presentation',
    'gender', 'region', 'highest_education', 'imd_band', 'age_band',
    'num_of_prev_attempts', 'studied_credits', 'disability',
    'final_result', 'date_registration', 'date_unregistration',
    'module_presentation_length'],
    how='left',
    suffixes=('', '_first')
)

# Step 3: Add binary column to indiciate who submitted first assessment (1=yes, 0=no)
df_first_assessment_full['submitted_first_assessment'] = (
    df_first_assessment_full['id_assessment'].notna().astype(int)
)

# Step 4: Merge in module-level first assessment info
df_first_assessment_full = df_first_assessment_full.merge(
    first_assessment[['code_module', 'code_presentation', 'id_assessment','assessment_type', 'due_date', 'weight']],
    on=['code_module', 'code_presentation'],
    how='left',
    suffixes=('', '_expected')
)

# Step 5: Use expected info for missing students
df_first_assessment_full['id_assessment'] = df_first_assessment_full['id_assessment'].fillna(
    df_first_assessment_full['id_assessment_expected']
)
df_first_assessment_full['assessment_type'] = df_first_assessment_full['assessment_type'].fillna(
    df_first_assessment_full['assessment_type_expected']
)
df_first_assessment_full['first_due_date'] = df_first_assessment_full['first_due_date'].fillna(
    df_first_assessment_full['due_date']
)
df_first_assessment_full['weight'] = df_first_assessment_full['weight'].fillna(
    df_first_assessment_full['weight_expected']
)

# Clean up helper columns
df_first_assessment_full.drop(columns=['id_assessment_expected','assessment_type_expected', 'due_date', 'weight_expected'], inplace=True)

df_first_assessment_full.head(10)

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,date,id_assessment,is_banked,score,assessment_type,first_due_date,weight,week,submitted_first_assessment
0,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,17.0,1758.0,0.0,60.0,TMA,19.0,10.0,3.0,1
1,8462,DDD,2013J,M,London Region,HE Qualification,30-40%,55<=,0,90,N,Withdrawn,-137.0,119.0,261,29.0,25348.0,0.0,93.0,TMA,25.0,10.0,5.0,1
2,8462,DDD,2014J,M,London Region,HE Qualification,30-40%,55<=,1,60,N,Withdrawn,-38.0,18.0,262,-1.0,25362.0,1.0,93.0,TMA,20.0,5.0,-1.0,1
3,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN,268,18.0,1752.0,0.0,78.0,TMA,19.0,10.0,3.0,1
4,23629,BBB,2013B,F,East Anglian Region,Lower Than A Level,20-30%,0-35,2,60,N,Fail,-47.0,NaN,240,9.0,14984.0,0.0,67.0,TMA,19.0,5.0,2.0,1
5,23698,CCC,2014J,F,East Anglian Region,A Level or Equivalent,50-60%,0-35,0,120,N,Pass,-110.0,NaN,269,21.0,24295.0,0.0,78.0,CMA,18.0,2.0,4.0,1
6,23798,BBB,2013J,M,Wales,A Level or Equivalent,50-60%,0-35,0,60,N,Distinction,-27.0,NaN,268,18.0,14996.0,0.0,90.0,TMA,19.0,5.0,3.0,1
7,24186,GGG,2014B,F,Yorkshire Region,Lower Than A Level,10-20,0-35,0,30,Y,Pass,-25.0,NaN,241,NaN,37425.0,NaN,NaN,TMA,61.0,0.0,NaN,0
8,24213,DDD,2014B,F,East Anglian Region,A Level or Equivalent,40-50%,0-35,1,60,N,Pass,-54.0,NaN,241,25.0,25355.0,0.0,78.0,TMA,25.0,10.0,4.0,1
9,24391,GGG,2013J,M,East Midlands Region,A Level or Equivalent,80-90%,0-35,0,30,N,Distinction,-64.0,NaN,261,57.0,37415.0,0.0,80.0,TMA,61.0,0.0,9.0,1


In [15]:
# Add submission related engineered features for FIRST ASSESSMENT only:
# - Relative submission date
# - Submission type (early, late, or never) - could add on-time if desired
df_first_assessment_full['relative_submission_date'] = df_first_assessment_full['first_due_date'] - df_first_assessment_full['date']
df_first_assessment_full['submission_type'] = np.select(
    [
        df_first_assessment_full['relative_submission_date'].isna(),
        df_first_assessment_full['relative_submission_date'] <= 0, # early & on-time
        # df_first_assessment_full['relative_submission_date'] == 0,
        df_first_assessment_full['relative_submission_date'] > 0
    ],
    [
        'Never',
        'Early',
        # 'On-time',
        'Late'
    ],
    default='Unknown'
)

df_first_assessment_full.head(10)

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,date,id_assessment,is_banked,score,assessment_type,first_due_date,weight,week,submitted_first_assessment,relative_submission_date,submission_type
0,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,17.0,1758.0,0.0,60.0,TMA,19.0,10.0,3.0,1,2.0,Late
1,8462,DDD,2013J,M,London Region,HE Qualification,30-40%,55<=,0,90,N,Withdrawn,-137.0,119.0,261,29.0,25348.0,0.0,93.0,TMA,25.0,10.0,5.0,1,-4.0,Early
2,8462,DDD,2014J,M,London Region,HE Qualification,30-40%,55<=,1,60,N,Withdrawn,-38.0,18.0,262,-1.0,25362.0,1.0,93.0,TMA,20.0,5.0,-1.0,1,21.0,Late
3,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN,268,18.0,1752.0,0.0,78.0,TMA,19.0,10.0,3.0,1,1.0,Late
4,23629,BBB,2013B,F,East Anglian Region,Lower Than A Level,20-30%,0-35,2,60,N,Fail,-47.0,NaN,240,9.0,14984.0,0.0,67.0,TMA,19.0,5.0,2.0,1,10.0,Late
5,23698,CCC,2014J,F,East Anglian Region,A Level or Equivalent,50-60%,0-35,0,120,N,Pass,-110.0,NaN,269,21.0,24295.0,0.0,78.0,CMA,18.0,2.0,4.0,1,-3.0,Early
6,23798,BBB,2013J,M,Wales,A Level or Equivalent,50-60%,0-35,0,60,N,Distinction,-27.0,NaN,268,18.0,14996.0,0.0,90.0,TMA,19.0,5.0,3.0,1,1.0,Late
7,24186,GGG,2014B,F,Yorkshire Region,Lower Than A Level,10-20,0-35,0,30,Y,Pass,-25.0,NaN,241,NaN,37425.0,NaN,NaN,TMA,61.0,0.0,NaN,0,NaN,Never
8,24213,DDD,2014B,F,East Anglian Region,A Level or Equivalent,40-50%,0-35,1,60,N,Pass,-54.0,NaN,241,25.0,25355.0,0.0,78.0,TMA,25.0,10.0,4.0,1,0.0,Early
9,24391,GGG,2013J,M,East Midlands Region,A Level or Equivalent,80-90%,0-35,0,30,N,Distinction,-64.0,NaN,261,57.0,37415.0,0.0,80.0,TMA,61.0,0.0,9.0,1,4.0,Late


In [16]:
# Sanity checks
student_module_presentation_combos = df_first_assessment_full[['id_student', 'code_module', 'code_presentation']].drop_duplicates()

print(student_module_presentation_combos.shape[0],"student - module - presentation combinations")
print(student_module_presentation_combos['id_student'].nunique(),"unique students")

print(df_first_assessment_full.shape)
print(df_first_assessment_full['submitted_first_assessment'].value_counts())
print(df_first_assessment_full['submission_type'].value_counts())

# proportion of students that submitted first assessment
df_first_assessment_full.groupby(['code_module', 'code_presentation'])['submitted_first_assessment'].mean()*100 

29228 student - module - presentation combinations
26074 unique students
(29228, 26)
submitted_first_assessment
1    25338
0     3890
Name: count, dtype: int64
submission_type
Late     14535
Early    10803
Never     3890
Name: count, dtype: int64


code_module  code_presentation
AAA          2013J                94.973545
             2014J                94.397759
BBB          2013B                87.898504
             2013J                89.946524
             2014B                91.499227
             2014J                92.243623
CCC          2014B                80.071386
             2014J                83.145091
DDD          2013B                83.031301
             2013J                84.106335
             2014B                83.243728
             2014J                85.731633
EEE          2013J                85.580913
             2014B                85.416667
             2014J                84.594348
FFF          2013B                90.132450
             2013J                88.560534
             2014B                87.454145
             2014J                85.808581
GGG          2013J                87.374302
             2014B                86.934023
             2014J                83.954155
N

Subset vle interaction data to the first 3 weeks of the course (including pre-course time period)

Summarize VLE interactions for first 3 weeks 

In [17]:
# Pre-course through week 3
df_vle_pre_w3 = df_vle[df_vle['week'] <= 3]
df_vle_pre_w3.head()

,id_student,code_module,code_presentation,date,forumng,homepage,oucontent,subpage,url,resource,glossary,dataplus,oucollaborate,quiz,ouelluminate,sharedsubpage,questionnaire,page,externalquiz,ouwiki,dualpane,repeatactivity,folder,htmlactivity,module_presentation_length,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,week
0,6516,AAA,2014J,-23.0,0,3,23,2,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,-4.0
1,6516,AAA,2014J,-22.0,33,13,34,0,0,2,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,-4.0
2,6516,AAA,2014J,-20.0,13,12,8,1,0,7,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,-3.0
3,6516,AAA,2014J,-17.0,0,2,0,3,2,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,-3.0
4,6516,AAA,2014J,-12.0,1,1,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,0,0,269,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,-2.0


In [18]:
# Check how many students got filtered out because they had zero vle interactions the first 3 weeks
print(df_vle[['id_student', 'code_module', 'code_presentation']].drop_duplicates().shape[0], "unique student - module - presentation combos in df_vle")

print(df_vle_pre_w3[['id_student', 'code_module', 'code_presentation']].drop_duplicates().shape[0], "unique student - module - presentation combos in df_vle_pre_w3")

# Now compute the differences
num_no_vle = (
    df_vle[['id_student', 'code_module', 'code_presentation']].drop_duplicates()
    .merge(df_vle_pre_w3[['id_student', 'code_module', 'code_presentation']].drop_duplicates(),
           on=['id_student', 'code_module', 'code_presentation'], how='left', indicator=True)
    .query('_merge == "left_only"')
    .shape[0]
)

print(num_no_vle, "student–module–presentation combinations that had no vle interactions in first 3 weeks.")

29228 unique student - module - presentation combos in df_vle
28567 unique student - module - presentation combos in df_vle_pre_w3
661 student–module–presentation combinations that had no vle interactions in first 3 weeks.


In [19]:
# Which students had no vle interactions? (useful for sanity checking code below)
# no_vle = (
#     df_vle[['id_student', 'code_module', 'code_presentation']].drop_duplicates()
#     .merge(df_vle_pre_w3[['id_student', 'code_module', 'code_presentation']].drop_duplicates(),
#            on=['id_student', 'code_module', 'code_presentation'], how='left', indicator=True)
#     .query('_merge == "left_only"')
# )
# no_vle.head()

In [20]:
# Add students who did not have vle interactions during first 3 weeks back into df_vle_pre_w3 and all vle columns will be 0:
# - date is assigned NaN	
# - week is assigned NaN

# Step 1: Start with all unique students from df_vle (keep all student info columns)
students_all = df_vle[[
    'id_student', 'code_module', 'code_presentation',
    'gender', 'region', 'highest_education', 'imd_band', 'age_band',
    'num_of_prev_attempts', 'studied_credits', 'disability',
    'final_result', 'date_registration', 'date_unregistration',
    'module_presentation_length'
]].drop_duplicates()

# Step 2: Merge with df_vle_pre_w3 (left join keeps all VLE students)
df_vle_pre_w3_full = students_all.merge(
    df_vle_pre_w3,
    on=['id_student', 'code_module', 'code_presentation',
    'gender', 'region', 'highest_education', 'imd_band', 'age_band',
    'num_of_prev_attempts', 'studied_credits', 'disability',
    'final_result', 'date_registration', 'date_unregistration',
    'module_presentation_length'],
    how='left',
    suffixes=('', '_first')
)

# Step 3: Assign 0 to all vle interactions for those students 
vle_columns = ['quiz', 'questionnaire', 'externalquiz', 'oucontent', 'page', 'resource', 'url', 'homepage', 
               'glossary', 'subpage', 'folder', 'forumng', 'oucollaborate', 'ouelluminate', 'ouwiki', 'sharedsubpage', 
               'dataplus', 'repeatactivity', 'dualpane', 'htmlactivity'
]
df_vle_pre_w3_full.loc[df_vle_pre_w3_full['date'].isna(), vle_columns] = 0

df_vle_pre_w3_full.head(10)

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,date,forumng,homepage,oucontent,subpage,url,resource,glossary,dataplus,oucollaborate,quiz,ouelluminate,sharedsubpage,questionnaire,page,externalquiz,ouwiki,dualpane,repeatactivity,folder,htmlactivity,week
0,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-23.0,0.0,3.0,23.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.0
1,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-22.0,33.0,13.0,34.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.0
2,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-20.0,13.0,12.0,8.0,1.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0
3,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-17.0,0.0,2.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0
4,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-12.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0
5,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-6.0,12.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0
6,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-5.0,0.0,7.0,5.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0
7,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-2.0,2.0,8.0,23.0,5.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0
8,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-1.0,3.0,7.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0
9,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,0.0,48.0,11.0,1.0,5.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [21]:
# Sanity check
df_vle_pre_w3_full[df_vle_pre_w3_full['id_student']==25997]

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,date,forumng,homepage,oucontent,subpage,url,resource,glossary,dataplus,oucollaborate,quiz,ouelluminate,sharedsubpage,questionnaire,page,externalquiz,ouwiki,dualpane,repeatactivity,folder,htmlactivity,week
182,25997,BBB,2014B,F,London Region,A Level or Equivalent,20-30%,0-35,2,60,N,Withdrawn,-139.0,83.0,234,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


In [22]:
# Sanity check
df_vle_pre_w3_full.loc[df_vle_pre_w3_full['date'].isna(), ['id_student', 'code_module', 'code_presentation']].drop_duplicates().shape[0]

661

In [23]:
# Add feature that sums total VLE interactions through week 3
# - total_vle_pre_w3

# Filter to pre-course through week 3 and sum interactions per student
student_totals = df_vle_pre_w3_full.groupby(['id_student','code_module','code_presentation'])[vle_columns].sum().reset_index()
student_totals['total_vle_pre_w3'] = student_totals[vle_columns].sum(axis=1)
student_totals.head()

,id_student,code_module,code_presentation,quiz,questionnaire,externalquiz,oucontent,page,resource,url,homepage,glossary,subpage,folder,forumng,oucollaborate,ouelluminate,ouwiki,sharedsubpage,dataplus,repeatactivity,dualpane,htmlactivity,total_vle_pre_w3
0,6516,AAA,2014J,0.0,0.0,0.0,245.0,0.0,15.0,31.0,120.0,0.0,37.0,0.0,158.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,606.0
1,8462,DDD,2013J,0.0,0.0,2.0,40.0,0.0,29.0,8.0,81.0,0.0,123.0,0.0,26.0,7.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,317.0
2,8462,DDD,2014J,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0
3,11391,AAA,2013J,0.0,0.0,0.0,273.0,0.0,9.0,1.0,42.0,0.0,21.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,401.0
4,23629,BBB,2013B,0.0,0.0,0.0,0.0,0.0,2.0,0.0,12.0,0.0,5.0,0.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,51.0


In [27]:
# Add VLE interaction category features:
# - Content type interaction (percentage)
# - Collaborative type interaction (percentage)

# This creates 2 columns for pre-course through week 3:
# - collaborative_focus_pre_w3
# - content_focus_pre_w3

content_types = ['oucontent', 'page', 'resource', 'url', 'homepage', 'glossary', 'subpage', 'folder']
collaborative_types = ['forumng', 'oucollaborate', 'ouelluminate', 'ouwiki', 'sharedsubpage']

student_totals['content_focus_pre_w3'] = student_totals[content_types].sum(axis=1) / student_totals['total_vle_pre_w3']
student_totals['content_focus_pre_w3'] = student_totals['content_focus_pre_w3'].fillna(0)

student_totals['collaborative_focus_pre_w3'] = student_totals[collaborative_types].sum(axis=1) / student_totals['total_vle_pre_w3']
student_totals['collaborative_focus_pre_w3'] = student_totals['collaborative_focus_pre_w3'].fillna(0)

In [28]:
# Add diversity of interaction features for pre-course through week 3
# - VLE richness (number of different VLE types used) 
# - Shannon entropy (overall diversity of interactions)

# This creates 2 columns:
# - vle_richness_pre_w3
# - diversity_shannon_pre_w3

 # Calculate richness (number of VLE types used)
student_totals['vle_richness_pre_w3'] = (student_totals[vle_columns]>0).sum(axis=1)

from scipy.stats import entropy

def shannon_entropy_calc(counts, vle_columns):
    counts = counts[vle_columns].values.astype(float)
    counts = counts[counts > 0]
    if len(counts) == 0:
        return 0
    proportions = counts / counts.sum()
    return entropy(proportions, base=2)

student_totals['diversity_shannon_pre_w3'] = student_totals[['id_student'] + vle_columns].apply(lambda row: shannon_entropy_calc(row, vle_columns), axis=1)

In [29]:
student_totals.head()

,id_student,code_module,code_presentation,quiz,questionnaire,externalquiz,oucontent,page,resource,url,homepage,glossary,subpage,folder,forumng,oucollaborate,ouelluminate,ouwiki,sharedsubpage,dataplus,repeatactivity,dualpane,htmlactivity,total_vle_pre_w3,content_focus_pre_w3,collaborative_focus_pre_w3,vle_richness_pre_w3,diversity_shannon_pre_w3
0,6516,AAA,2014J,0.0,0.0,0.0,245.0,0.0,15.0,31.0,120.0,0.0,37.0,0.0,158.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,606.0,0.739274,0.260726,6,2.094273
1,8462,DDD,2013J,0.0,0.0,2.0,40.0,0.0,29.0,8.0,81.0,0.0,123.0,0.0,26.0,7.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,317.0,0.886435,0.107256,9,2.349100
2,8462,DDD,2014J,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.700000,0.300000,3,1.156780
3,11391,AAA,2013J,0.0,0.0,0.0,273.0,0.0,9.0,1.0,42.0,0.0,21.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,401.0,0.862843,0.137157,6,1.479023
4,23629,BBB,2013B,0.0,0.0,0.0,0.0,0.0,2.0,0.0,12.0,0.0,5.0,0.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,51.0,0.372549,0.627451,4,1.424794


In [30]:
# Merge these features back into dataframe
df_vle_pre_w3_full = df_vle_pre_w3_full.merge(
    student_totals[['id_student', 'code_module','code_presentation','total_vle_pre_w3','content_focus_pre_w3','collaborative_focus_pre_w3','vle_richness_pre_w3','diversity_shannon_pre_w3']],
    on=['id_student', 'code_module','code_presentation'],
    how='left'
)
df_vle_pre_w3_full.head()

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,date,forumng,homepage,oucontent,subpage,url,resource,glossary,dataplus,oucollaborate,quiz,ouelluminate,sharedsubpage,questionnaire,page,externalquiz,ouwiki,dualpane,repeatactivity,folder,htmlactivity,week,total_vle_pre_w3,content_focus_pre_w3,collaborative_focus_pre_w3,vle_richness_pre_w3,diversity_shannon_pre_w3
0,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-23.0,0.0,3.0,23.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.0,606.0,0.739274,0.260726,6,2.094273
1,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-22.0,33.0,13.0,34.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.0,606.0,0.739274,0.260726,6,2.094273
2,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-20.0,13.0,12.0,8.0,1.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,606.0,0.739274,0.260726,6,2.094273
3,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-17.0,0.0,2.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,606.0,0.739274,0.260726,6,2.094273
4,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-12.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,606.0,0.739274,0.260726,6,2.094273


In [31]:
# Add regularity features for pre-course through week 3:
# - Standard deviation of active days per week
# - Standard deviation of gaps between days with interactions

# This creates 2 columns:
# - active_days_per_week_pre_w3
# - std_regularity_pre_w3

def calculate_weekly_consistency(df_filtered, suffix, student_id_col='id_student', date_col='date', week_col='week'):
    weekly_days = df_filtered.groupby([student_id_col, week_col])[date_col].nunique()
    consistency = weekly_days.groupby(student_id_col).std()
    consistency = consistency.rename(f'active_days_per_week_pre_w3_{suffix}')
    return consistency

def calculate_regularity_std_period(df_filtered, suffix, student_id_col='id_student', date_col='date'):
    regularity = df_filtered.groupby(student_id_col)[date_col].apply(
        lambda x: x.sort_values().diff().std()
    )
    regularity = regularity.rename(f'std_regularity_{suffix}')
    return regularity

active_days_pre_w3 = calculate_weekly_consistency(df_vle_pre_w3_full, 'pre_w3')
active_days_pre_w3

std_regularity_pre_w3 = calculate_regularity_std_period(df_vle_pre_w3_full, 'pre_w3')
std_regularity_pre_w3 

# Merge both metrics back to original dataframe
df_vle_pre_w3_full['active_days_per_week_pre_w3'] = df_vle_pre_w3_full['id_student'].map(active_days_pre_w3)
df_vle_pre_w3_full['std_regularity_pre_w3'] = df_vle_pre_w3_full['id_student'].map(std_regularity_pre_w3)

df_vle_pre_w3_full.head()

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,date,forumng,homepage,oucontent,subpage,url,resource,glossary,dataplus,oucollaborate,quiz,ouelluminate,sharedsubpage,questionnaire,page,externalquiz,ouwiki,dualpane,repeatactivity,folder,htmlactivity,week,total_vle_pre_w3,content_focus_pre_w3,collaborative_focus_pre_w3,vle_richness_pre_w3,diversity_shannon_pre_w3,active_days_per_week_pre_w3,std_regularity_pre_w3
0,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-23.0,0.0,3.0,23.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.0,606.0,0.739274,0.260726,6,2.094273,1.812654,1.358621
1,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-22.0,33.0,13.0,34.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.0,606.0,0.739274,0.260726,6,2.094273,1.812654,1.358621
2,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-20.0,13.0,12.0,8.0,1.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,606.0,0.739274,0.260726,6,2.094273,1.812654,1.358621
3,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-17.0,0.0,2.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,606.0,0.739274,0.260726,6,2.094273,1.812654,1.358621
4,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,-12.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,606.0,0.739274,0.260726,6,2.094273,1.812654,1.358621


In [32]:
# check a student who had no vle interactions
df_vle_pre_w3_full[df_vle_pre_w3_full['id_student']==25997]
# - total_vle_pre_w3 = 0
# - content_focus_pre_w3 = 0
# - collaborative_focus_pre_w3 = 0
# - vle_richness_pre_w3 = 0
# - diversity_shannon_pre_w3 = 0
# - weekly_consistency_pre_w3 = NaN
# - std_regularity_pre_w3 = NaN

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,date,forumng,homepage,oucontent,subpage,url,resource,glossary,dataplus,oucollaborate,quiz,ouelluminate,sharedsubpage,questionnaire,page,externalquiz,ouwiki,dualpane,repeatactivity,folder,htmlactivity,week,total_vle_pre_w3,content_focus_pre_w3,collaborative_focus_pre_w3,vle_richness_pre_w3,diversity_shannon_pre_w3,active_days_per_week_pre_w3,std_regularity_pre_w3
182,25997,BBB,2014B,F,London Region,A Level or Equivalent,20-30%,0-35,2,60,N,Withdrawn,-139.0,83.0,234,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0,0.0,NaN,NaN


In [33]:
# Collapse dataframe to one row per student
# Group by student-module-presentation, taking the first value for each column
# (since student-level features should be the same across all rows for a given student)
df_vle_pre_w3_student_level = df_vle_pre_w3_full.groupby(['id_student','code_module','code_presentation']).first().reset_index()
df_vle_pre_w3_student_level = df_vle_pre_w3_student_level.drop(columns=['date', 'week'] + vle_columns)

df_vle_pre_w3_student_level.head()

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,total_vle_pre_w3,content_focus_pre_w3,collaborative_focus_pre_w3,vle_richness_pre_w3,diversity_shannon_pre_w3,active_days_per_week_pre_w3,std_regularity_pre_w3
0,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,606.0,0.739274,0.260726,6,2.094273,1.812654,1.358621
1,8462,DDD,2013J,M,London Region,HE Qualification,30-40%,55<=,0,90,N,Withdrawn,-137.0,119.0,261,317.0,0.886435,0.107256,9,2.349100,1.414214,0.704154
2,8462,DDD,2014J,M,London Region,HE Qualification,30-40%,55<=,1,60,N,Withdrawn,-38.0,18.0,262,10.0,0.700000,0.300000,3,1.156780,1.414214,0.704154
3,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN,268,401.0,0.862843,0.137157,6,1.479023,1.414214,2.627691
4,23629,BBB,2013B,F,East Anglian Region,Lower Than A Level,20-30%,0-35,2,60,N,Fail,-47.0,NaN,240,51.0,0.372549,0.627451,4,1.424794,0.500000,2.872281


Re-combine vle interactions and assessments dataframe

In [34]:
# Sanity check
print(df_vle_pre_w3_student_level[['id_student', 'code_module', 'code_presentation']].drop_duplicates().shape[0], "unique student - module - presentation combos in df_vle_pre_w3_student_level")
print(df_first_assessment_full[['id_student', 'code_module', 'code_presentation']].drop_duplicates().shape[0], "unique student - module - presentation combos in df_first_assessment_full")

29228 unique student - module - presentation combos in df_vle_pre_w3_student_level
29228 unique student - module - presentation combos in df_first_assessment_full


In [35]:
common_columns = list(set(df_vle_pre_w3_student_level.columns) & set(df_first_assessment_full.columns)) 
df_pre_w3=pd.merge(df_vle_pre_w3_student_level, df_first_assessment_full, how='left', on=common_columns)

In [36]:
print(df_pre_w3[['id_student', 'code_module', 'code_presentation']].drop_duplicates().shape[0], "unique student - module - presentation combos in df_pre_w3")
df_pre_w3.head()

29228 unique student - module - presentation combos in df_pre_w3


,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,total_vle_pre_w3,content_focus_pre_w3,collaborative_focus_pre_w3,vle_richness_pre_w3,diversity_shannon_pre_w3,active_days_per_week_pre_w3,std_regularity_pre_w3,date,id_assessment,is_banked,score,assessment_type,first_due_date,weight,week,submitted_first_assessment,relative_submission_date,submission_type
0,6516,AAA,2014J,M,Scotland,HE Qualification,80-90%,55<=,0,60,N,Pass,-52.0,NaN,269,606.0,0.739274,0.260726,6,2.094273,1.812654,1.358621,17.0,1758.0,0.0,60.0,TMA,19.0,10.0,3.0,1,2.0,Late
1,8462,DDD,2013J,M,London Region,HE Qualification,30-40%,55<=,0,90,N,Withdrawn,-137.0,119.0,261,317.0,0.886435,0.107256,9,2.349100,1.414214,0.704154,29.0,25348.0,0.0,93.0,TMA,25.0,10.0,5.0,1,-4.0,Early
2,8462,DDD,2014J,M,London Region,HE Qualification,30-40%,55<=,1,60,N,Withdrawn,-38.0,18.0,262,10.0,0.700000,0.300000,3,1.156780,1.414214,0.704154,-1.0,25362.0,1.0,93.0,TMA,20.0,5.0,-1.0,1,21.0,Late
3,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN,268,401.0,0.862843,0.137157,6,1.479023,1.414214,2.627691,18.0,1752.0,0.0,78.0,TMA,19.0,10.0,3.0,1,1.0,Late
4,23629,BBB,2013B,F,East Anglian Region,Lower Than A Level,20-30%,0-35,2,60,N,Fail,-47.0,NaN,240,51.0,0.372549,0.627451,4,1.424794,0.500000,2.872281,9.0,14984.0,0.0,67.0,TMA,19.0,5.0,2.0,1,10.0,Late


Subset students to exclude <br>
- students who have taken the course before (num_prev_attemps>0) <br>
- students who withdrew before week 4 (final_result==Withdrawn & date_unregistration<=20) <br>

In [37]:
# Dropping rows in which a student already took the course
df_pre_w3_subset = df_pre_w3[df_pre_w3['num_of_prev_attempts'] == 0]

In [38]:
# Drop students who withdrew before week 4 (i.e., withdrew during weeks 1-3 or pre-course)
df_pre_w3_subset = df_pre_w3_subset[~((df_pre_w3_subset['final_result'] == 'Withdrawn') & (df_pre_w3_subset['date_unregistration'] <= 20))]

In [39]:
# Check how many students were excluded 
# Count total students per module-presentation
total_students_per_modpres = (
    df_pre_w3.groupby(['code_module', 'code_presentation'])['id_student']
    .nunique()
    .reset_index(name='total_students')
)

# Count only students retained after filtering
subset_students_per_modpres = (
    df_pre_w3_subset.groupby(['code_module', 'code_presentation'])['id_student']
    .nunique()
    .reset_index(name='subset_students')
)

# Merge and compute percentage
excluded_summary = total_students_per_modpres.merge(
    subset_students_per_modpres,
    on=['code_module', 'code_presentation'],
    how='left'
)

# Fill missing counts with 0 and compute percentage
excluded_summary['pct_retained_student'] = (
    excluded_summary['subset_students'] / excluded_summary['total_students'] * 100
)

# Show result
print(excluded_summary)

   code_module code_presentation  total_students  subset_students  \
0          AAA             2013J             378              372   
1          AAA             2014J             357              310   
2          BBB             2013B            1537             1212   
3          BBB             2013J            1870             1518   
4          BBB             2014B            1294             1043   
5          BBB             2014J            1921             1560   
6          CCC             2014B            1681             1515   
7          CCC             2014J            2302             1843   
8          DDD             2013B            1214              923   
9          DDD             2013J            1768             1400   
10         DDD             2014B            1116              803   
11         DDD             2014J            1647             1194   
12         EEE             2013J             964              914   
13         EEE             2014B  

In [41]:
# Check how many students retained in each target category 
# # Count unique students per final_result and per module-presentation
final_result_counts = (
    df_pre_w3_subset
    .drop_duplicates(subset=['id_student', 'code_module', 'code_presentation'])
    .groupby(['code_module', 'code_presentation', 'final_result'])['id_student']
    .nunique()
    .reset_index(name='num_unique_students')
)

# Pivot so each final_result becomes its own column
final_result_pivot = (
    final_result_counts
    .pivot(index=['code_module', 'code_presentation'],
           columns='final_result',
           values='num_unique_students')
    .fillna(0)  # fill missing combinations with 0
    .astype(int) # optional: make counts integers
    .reset_index()
)

print(final_result_pivot)


final_result code_module code_presentation  Distinction  Fail  Pass  Withdrawn
0                    AAA             2013J           20    45   258         49
1                    AAA             2014J           23    39   207         41
2                    BBB             2013B          141   306   542        223
3                    BBB             2013J          160   371   800        187
4                    BBB             2014B          151   278   485        129
5                    BBB             2014J          167   313   871        209
6                    CCC             2014B          192   355   471        497
7                    CCC             2014J          299   344   650        550
8                    DDD             2013B           48   256   390        229
9                    DDD             2013J           91   325   647        337
10                   DDD             2014B          104   174   303        222
11                   DDD             2014J          

Subset data to exclude
- modules that do not have a first assessment in the first 3 weeks

In [42]:
# Create a folder to store the CSV files (optional but recommended)
output_folder = "modules_csv"
os.makedirs(output_folder, exist_ok=True)

# Filter to only include modules that have a first assessment in the first 3 weeks of the course
selected_modules = modules_to_keep['code_module'].drop_duplicates().to_list() # ['AAA', 'BBB', 'CCC']
df_filtered = df_pre_w3_subset[df_pre_w3_subset['code_module'].isin(selected_modules)]

# Save all selected modules together in a single file
filename = f"{output_folder}/combined_modules_data_w3.csv"
df_filtered.to_csv(filename, index=False)
print(f"Saved: {filename}")
print(f"Included modules: {', '.join(selected_modules)}")
print(f"Total rows: {len(df_filtered)}")
print(f"Total unique students: {df_filtered['id_student'].nunique()}")

Saved: modules_csv/combined_modules_data_w3.csv
Included modules: AAA, BBB, CCC
Total rows: 9373
Total unique students: 9373
